In [ ]:
# Environment
try:
    import pyarrow  # noqa
except ImportError:
    !pip -q install pyarrow
try:
    import geopandas  # noqa
except ImportError:
    !pip -q install geopandas

import os, io, time, requests
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

pd.set_option("display.width", 120)
print("pandas", pd.__version__, "| pyarrow", pyarrow.__version__)

pandas 2.2.2 | pyarrow 18.1.0


In [ ]:
# Configuration
START_YM = (2023, 1)     # first month (inclusive)
END_YM   = (2025, 12)    # last month to *attempt*; missing months are skipped automatically
SERVICE  = "yellow"      # step 1 uses yellow taxi only (cleanest, Manhattan-heavy).
                         # HVFHV (where the $1.50 fee & most CRZ trips sit) is a documented
                         # extension: far larger files, same pipeline.

TLC_BASE   = "https://d37ci6vzurychx.cloudfront.net"
TRIP_URL   = TLC_BASE + "/trip-data/{svc}_tripdata_{y}-{m:02d}.parquet"
LOOKUP_URL = TLC_BASE + "/misc/taxi_zone_lookup.csv"
# Authoritative CRZ taxi-zone set, published by the MTA for exactly this purpose
CRZ_ZONES_URL = "https://data.ny.gov/api/views/yfdc-w5jh/rows.csv?accessType=DOWNLOAD"
# Taxi-zone geometry (official TLC shapefile), used to derive the buffer ring
GEOM_URL  = TLC_BASE + "/misc/taxi_zones.zip"
BUFFER_FT = 100          # zones within 100 ft of the cordon = land-adjacent (rivers are wider)

# Resolve a persistent DATA_DIR (Drive on Google Colab, else local)
def resolve_data_dir():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        d = "/content/drive/MyDrive/nyc-crz-counterfactual"
    except Exception:
        d = "./nyc-crz-counterfactual"
    os.makedirs(d, exist_ok=True)
    return d

DATA_DIR    = resolve_data_dir()
MONTHLY_DIR = os.path.join(DATA_DIR, "monthly_3grp")
os.makedirs(MONTHLY_DIR, exist_ok=True)
PANEL_PATH  = os.path.join(DATA_DIR, f"crz_hourly_panel_{SERVICE}_3grp.parquet")
RAW_TMP     = "/content/_raw_tmp.parquet" if os.path.isdir("/content") else "./_raw_tmp.parquet"
print("DATA_DIR =", DATA_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DATA_DIR = /content/drive/MyDrive/nyc-crz-counterfactual


In [ ]:
# CRZ definition: official MTA CBD Taxi Zones (authoritative)
# Hand-derived list kept only as a cross-check / offline fallback.
CRZ_ZONE_NAMES_FALLBACK = {
    "Battery Park", "Battery Park City", "World Trade Center", "Seaport",
    "Financial District North", "Financial District South",
    "TriBeCa/Civic Center", "SoHo", "Little Italy/NoLiTa", "Chinatown",
    "Lower East Side", "Two Bridges/Seward Park", "Alphabet City", "East Village",
    "Greenwich Village North", "Greenwich Village South", "West Village", "Hudson Sq",
    "Meatpacking/West Village West", "East Chelsea", "West Chelsea/Hudson Yards",
    "Flatiron", "Union Sq", "Gramercy", "Stuy Town/Peter Cooper Village",
    "Garment District", "Penn Station/Madison Sq West", "Times Sq/Theatre District",
    "Murray Hill", "Kips Bay", "Midtown South", "Midtown Center", "Midtown East",
    "Midtown North", "Clinton East", "Clinton West",
    "Sutton Place/Turtle Bay North", "UN/Turtle Bay South",
}

def _norm(s):
    import re
    s = re.sub(r"\s*/\s*", "/", str(s).strip().lower())
    return re.sub(r"\s+", " ", s)

def parse_crz_ids(crz_raw, zone_lookup):
    """Extract CRZ LocationIDs from the official MTA file, robust to (a) its
    exact column name and (b) whether that column holds ID numbers or zone names."""
    norm_cols = {c: c.lower().replace(" ", "").replace("_", "") for c in crz_raw.columns}
    col = None
    for cand in ("locationid", "objectid", "taxizone", "zone"):
        for c, n in norm_cols.items():
            if n == cand:
                col = c; break
        if col: break
    if col is None:
        for c, n in norm_cols.items():
            if ("location" in n and "id" in n) or ("taxi" in n and "zone" in n):
                col = c; break
    if col is None:
        raise ValueError(f"No zone column in {list(crz_raw.columns)}")

    vals    = crz_raw[col].dropna()
    as_num  = pd.to_numeric(vals, errors="coerce").dropna().astype(int)
    if len(as_num) >= 0.5 * len(vals) and as_num.between(1, 263).mean() > 0.9:
        return set(as_num[as_num.between(1, 263)]), f"numeric ids in '{col}'", set()

    norm_to_id = {_norm(z): i for z, i in zip(zone_lookup["Zone"], zone_lookup["LocationID"])}
    ids, unmatched = set(), set()
    for v in vals.unique():
        key = _norm(v)
        (ids.add(int(norm_to_id[key])) if key in norm_to_id else unmatched.add(v))
    return ids, f"names in '{col}'", unmatched

def ids_from_names(names, zone_lookup):
    name_to_id = dict(zip(zone_lookup["Zone"], zone_lookup["LocationID"]))
    return ({name_to_id[n] for n in names if n in name_to_id},
            {n for n in names if n not in name_to_id})

zones    = pd.read_csv(LOOKUP_URL)
id2name  = dict(zip(zones["LocationID"], zones["Zone"]))

try:
    crz_raw = pd.read_csv(CRZ_ZONES_URL)
    CRZ_IDS, method, unmatched_official = parse_crz_ids(crz_raw, zones)
    if unmatched_official:
        print("⚠️ Official zone names not found in TLC lookup (check spelling/version):")
        for n in sorted(unmatched_official):
            print("   -", n)
    SOURCE = f"official MTA CBD Taxi Zones ({method}, {len(CRZ_IDS)} zones)"
except Exception as e:
    print("⚠️ Official CRZ fetch/parse failed:", repr(e), "\n   -> using hand-derived fallback list.")
    CRZ_IDS, _ = ids_from_names(CRZ_ZONE_NAMES_FALLBACK, zones)
    SOURCE = f"hand-derived fallback ({len(CRZ_IDS)} zones)"

print("CRZ source:", SOURCE)
for i in sorted(CRZ_IDS):
    print(f"  {i:>3}  {id2name.get(i, '??? (not in TLC lookup)')}")

CRZ source: official MTA CBD Taxi Zones (numeric ids in 'Taxi Zone', 38 zones)
    4  Alphabet City
   12  Battery Park
   13  Battery Park City
   45  Chinatown
   48  Clinton East
   50  Clinton West
   68  East Chelsea
   79  East Village
   87  Financial District North
   88  Financial District South
   90  Flatiron
  100  Garment District
  107  Gramercy
  113  Greenwich Village North
  114  Greenwich Village South
  125  Hudson Sq
  137  Kips Bay
  144  Little Italy/NoLiTa
  148  Lower East Side
  158  Meatpacking/West Village West
  161  Midtown Center
  162  Midtown East
  163  Midtown North
  164  Midtown South
  170  Murray Hill
  186  Penn Station/Madison Sq West
  209  Seaport
  211  SoHo
  224  Stuy Town/Peter Cooper Village
  229  Sutton Place/Turtle Bay North
  230  Times Sq/Theatre District
  231  TriBeCa/Civic Center
  232  Two Bridges/Seward Park
  233  UN/Turtle Bay South
  234  Union Sq
  246  West Chelsea/Hudson Yards
  249  West Village
  261  World Trade Center


In [ ]:
# Buffer ring: non-CRZ zones land-adjacent to the cordon
import io
import geopandas as gpd

# Defined here too, so this cell runs standalone even if the config cell wasn't re-run
GEOM_URLS = [
    "https://data.source.coop/cholmes/nyc-taxi-zones/taxi_zones.parquet",       # EPSG:2263 (ft)
    "https://source.coop/cholmes/nyc-taxi-zones/taxi_zones.parquet",
    "https://data.source.coop/cholmes/nyc-taxi-zones/taxi_zones_4326.parquet",  # lon/lat fallback
]
BUFFER_FT = 100

def load_zone_geometry(urls):
    last = None
    for u in urls:
        try:
            r = requests.get(u, timeout=120); r.raise_for_status()
            gdf = gpd.read_parquet(io.BytesIO(r.content))
            if "LocationID" not in gdf.columns:            # normalise id column name
                for c in gdf.columns:
                    if c.lower().replace("_", "") == "locationid":
                        gdf = gdf.rename(columns={c: "LocationID"}); break
            gdf["LocationID"] = gdf["LocationID"].astype(int)
            crs = gdf.crs.to_epsg() if gdf.crs else "?"
            print(f"Loaded geometry: {len(gdf)} zones, CRS {crs}  [{u}]")
            return gdf
        except Exception as e:
            last = e; print(f"  source failed ({u}): {e!r}")
    raise RuntimeError(f"All geometry sources failed; last error: {last!r}")

def compute_buffer_ids(gdf, crz_ids, eps_ft=BUFFER_FT):
    if gdf.crs is None:               g = gdf.set_crs(2263, allow_override=True)
    elif gdf.crs.to_epsg() != 2263:   g = gdf.to_crs(2263)   # planar feet for buffering
    else:                             g = gdf
    ring = g[g["LocationID"].isin(crz_ids)].geometry.union_all().buffer(eps_ft)
    cand = g[~g["LocationID"].isin(crz_ids)]
    return set(int(i) for i in cand[cand.geometry.intersects(ring)]["LocationID"])

gdf        = load_zone_geometry(GEOM_URLS)
BUFFER_IDS = compute_buffer_ids(gdf, CRZ_IDS) - CRZ_IDS
print(f"\nBuffer ring: {len(BUFFER_IDS)} zones within {BUFFER_FT} ft of the cordon")
for i in sorted(BUFFER_IDS):
    print(f"  {i:>3}  {id2name.get(i, '???')}")

Loaded geometry: 263 zones, CRS 2263  [https://data.source.coop/cholmes/nyc-taxi-zones/taxi_zones.parquet]

Buffer ring: 6 zones within 100 ft of the cordon
   43  Central Park
  140  Lenox Hill East
  141  Lenox Hill West
  142  Lincoln Square East
  143  Lincoln Square West
  237  Upper East Side South


In [ ]:
# Cleaning / tagging / aggregation (tested)
MAX_DURATION_MIN, MAX_DISTANCE_MI, MAX_SPEED_MPH = 180.0, 100.0, 80.0
UNKNOWN_ZONE_IDS = {264, 265}
NEEDED_COLS = ["tpep_pickup_datetime", "tpep_dropoff_datetime",
               "PULocationID", "DOLocationID", "trip_distance"]

def clean_and_tag(df, month_start, crz_ids, buffer_ids):
    next_month = month_start + pd.offsets.MonthBegin(1)
    d = df.rename(columns={"tpep_pickup_datetime": "pickup",
                           "tpep_dropoff_datetime": "dropoff"})
    d = d[["pickup", "dropoff", "PULocationID", "DOLocationID", "trip_distance"]].copy()
    d["pickup"]  = pd.to_datetime(d["pickup"], errors="coerce")
    d["dropoff"] = pd.to_datetime(d["dropoff"], errors="coerce")
    d = d.dropna(subset=["pickup", "dropoff", "PULocationID", "DOLocationID"])
    d = d[~d["PULocationID"].isin(UNKNOWN_ZONE_IDS) & ~d["DOLocationID"].isin(UNKNOWN_ZONE_IDS)]
    d = d[(d["pickup"] >= month_start) & (d["pickup"] < next_month)]
    d["duration_min"] = (d["dropoff"] - d["pickup"]).dt.total_seconds() / 60.0
    d = d[(d["duration_min"] > 0) & (d["duration_min"] <= MAX_DURATION_MIN)]
    d = d[(d["trip_distance"] > 0) & (d["trip_distance"] <= MAX_DISTANCE_MI)]
    d["speed_mph"] = d["trip_distance"] / (d["duration_min"] / 60.0)
    d = d[(d["speed_mph"] > 0) & (d["speed_mph"] <= MAX_SPEED_MPH)]
    if d.empty:
        return d.assign(group=pd.Series(dtype="object"), hour=pd.Series(dtype="datetime64[ns]"))
    treated = d["PULocationID"].isin(crz_ids)    | d["DOLocationID"].isin(crz_ids)
    buf     = d["PULocationID"].isin(buffer_ids) | d["DOLocationID"].isin(buffer_ids)
    # precedence: touching the CRZ -> treated; else touching the ring -> buffer; else control
    d["group"] = np.select([treated, (~treated) & buf], ["treated", "buffer"], default="control")
    d["hour"]  = d["pickup"].dt.floor("h")
    return d

def aggregate_hourly(tagged):
    cols = ["datetime","group","n_trips","dur_mean_min","dur_median_min",
            "speed_mean_mph","speed_median_mph"]
    if tagged.empty:
        return pd.DataFrame(columns=cols)
    g = tagged.groupby(["hour","group"]).agg(
        n_trips=("duration_min","size"),
        dur_mean_min=("duration_min","mean"),
        dur_median_min=("duration_min","median"),
        speed_mean_mph=("speed_mph","mean"),
        speed_median_mph=("speed_mph","median"),
    ).reset_index().rename(columns={"hour":"datetime"})
    return g

In [ ]:
# Resumable month loop
def month_range(start, end):
    y, m = start
    while (y, m) <= end:
        yield y, m
        m += 1
        if m == 13:
            y, m = y + 1, 1

def fetch_and_process(y, m):
    out = os.path.join(MONTHLY_DIR, f"{SERVICE}_{y}-{m:02d}.parquet")
    if os.path.exists(out):
        return "cached"
    url = TRIP_URL.format(svc=SERVICE, y=y, m=m)
    r = requests.get(url, stream=True, timeout=120)
    if r.status_code == 404:
        return "missing"
    r.raise_for_status()
    with open(RAW_TMP, "wb") as f:
        for chunk in r.iter_content(chunk_size=1 << 20):
            f.write(chunk)
    df = pq.read_table(RAW_TMP, columns=NEEDED_COLS).to_pandas()
    panel = aggregate_hourly(clean_and_tag(df, pd.Timestamp(y, m, 1), CRZ_IDS, BUFFER_IDS))
    panel.to_parquet(out, index=False)
    del df
    if os.path.exists(RAW_TMP):
        os.remove(RAW_TMP)
    return f"{len(panel)} rows"

for y, m in month_range(START_YM, END_YM):
    t0 = time.time()
    try:
        status = fetch_and_process(y, m)
    except Exception as e:
        status = f"ERROR: {e}"
    print(f"{y}-{m:02d}: {status:>14}   ({time.time()-t0:5.1f}s)")

2023-01:      2232 rows   (  2.3s)
2023-02:      2016 rows   (  2.1s)
2023-03:      2229 rows   (  3.7s)
2023-04:      2160 rows   (  4.3s)
2023-05:      2232 rows   (  2.1s)
2023-06:      2160 rows   (  2.3s)
2023-07:      2232 rows   (  2.0s)
2023-08:      2232 rows   (  1.7s)
2023-09:      2158 rows   (  2.3s)
2023-10:      2232 rows   (  2.2s)
2023-11:      2160 rows   (  2.0s)
2023-12:      2232 rows   (  2.0s)
2024-01:      2232 rows   (  3.3s)
2024-02:      2088 rows   (  3.9s)
2024-03:      2229 rows   (  4.0s)
2024-04:      2160 rows   (  3.7s)
2024-05:      2232 rows   (  5.3s)
2024-06:      2160 rows   (  5.6s)
2024-07:      2232 rows   (  2.5s)
2024-08:      2232 rows   (  4.0s)
2024-09:      2160 rows   (  4.5s)
2024-10:      2232 rows   (  2.6s)
2024-11:      2160 rows   (  2.2s)
2024-12:      2232 rows   (  2.8s)
2025-01:      2232 rows   (  2.1s)
2025-02:      2016 rows   (  2.4s)
2025-03:      2229 rows   (  2.7s)
2025-04:      2160 rows   (  2.0s)
2025-05:      2232 r

In [ ]:
# Combine + cache final panel
parts = [pd.read_parquet(os.path.join(MONTHLY_DIR, f))
         for f in sorted(os.listdir(MONTHLY_DIR)) if f.startswith(SERVICE)]
panel = (pd.concat(parts, ignore_index=True)
           .sort_values(["datetime", "group"])
           .reset_index(drop=True))
panel.to_parquet(PANEL_PATH, index=False)
print("Saved:", PANEL_PATH, "| shape:", panel.shape)

Saved: /content/drive/MyDrive/nyc-crz-counterfactual/crz_hourly_panel_yellow_3grp.parquet | shape: (78901, 7)


In [ ]:
# Validation
print("Date coverage :", panel.datetime.min(), "->", panel.datetime.max())
print("Groups        :", panel.group.value_counts().to_dict())
print("Missing values:\n", panel.isna().sum().to_string())

# Hourly completeness: expected up to 3 rows (treated+buffer+control) per hour
hours = panel.datetime.nunique()
span_hours = int((panel.datetime.max() - panel.datetime.min()).total_seconds() // 3600) + 1
print(f"\nDistinct hours present: {hours} / {span_hours} in span "
      f"({100*hours/span_hours:.1f}%)")

# Daily volume around the policy date (sanity, not analysis)
daily = (panel.assign(date=panel.datetime.dt.date)
              .groupby(["date","group"])["n_trips"].sum().unstack("group"))
print("\nDaily trip volume around 2025-01-05:")
print(daily.loc[pd.to_datetime("2024-12-30").date():pd.to_datetime("2025-01-09").date()]
          .to_string())

panel.head()

Date coverage : 2023-01-01 00:00:00 -> 2025-12-31 23:00:00
Groups        : {'control': 26301, 'treated': 26301, 'buffer': 26299}
Missing values:
 datetime            0
group               0
n_trips             0
dur_mean_min        0
dur_median_min      0
speed_mean_mph      0
speed_median_mph    0

Distinct hours present: 26301 / 26304 in span (100.0%)

Daily trip volume around 2025-01-05:
group       buffer  control  treated
date                                
2024-12-30   10292    12120    56896
2024-12-31   12809    14962    58039
2025-01-01    8659    15291    63739
2025-01-02   11208    12150    58989
2025-01-03   12356    12943    63263
2025-01-04   12355    12537    70774
2025-01-05    9835    12615    54970
2025-01-06   12978    12799    52047
2025-01-07   16158    14210    67101
2025-01-08   17041    15023    77871
2025-01-09   17791    15522    81443


,datetime,group,n_trips,dur_mean_min,dur_median_min,speed_mean_mph,speed_median_mph
0,2023-01-01 00:00:00,buffer,691,9.761433,7.783333,13.649953,12.558140
1,2023-01-01 00:00:00,control,745,14.740984,11.150000,19.083654,15.267857
2,2023-01-01 00:00:00,treated,3713,16.021901,13.933333,11.414389,10.396484
3,2023-01-01 01:00:00,buffer,703,10.345590,8.083333,13.714391,12.875536
4,2023-01-01 01:00:00,control,853,14.210121,10.600000,16.135829,13.141593


In [ ]:
panel["date"] = panel["datetime"].dt.date
daily = panel.groupby([panel.datetime.dt.to_period("M").astype(str), "group"]).agg(
    trips=("n_trips","sum"),
    dur=("dur_median_min","mean"),
    spd=("speed_median_mph","mean")).round(2)
print(daily.to_string())

                    trips    dur    spd
datetime group                         
2023-01  buffer    424404   8.68  12.56
         control   389616  12.26  17.17
         treated  2148650  11.66  11.78
2023-02  buffer    383398   8.67  12.40
         control   349877  12.23  16.39
         treated  2086247  11.95  11.41
2023-03  buffer    437031   8.83  12.33
         control   414273  12.93  16.59
         treated  2450462  12.38  11.21
2023-04  buffer    428990   8.99  12.26
         control   408084  13.21  16.49
         treated  2356870  12.52  11.36
2023-05  buffer    467632   9.52  11.97
         control   438002  13.69  16.19
         treated  2503868  13.10  11.11
2023-06  buffer    415673   9.22  12.11
         control   408226  14.26  16.76
         treated  2380224  12.95  11.04
2023-07  buffer    324364   8.74  12.75
         control   352691  14.87  18.34
         treated  2129332  12.40  11.30
2023-08  buffer    312279   8.69  12.45
         control   352486  15.18  18.08
